# 08 — Figures, tables, paper population, HF hub push

Reads training logs + eval results, generates publication-ready figures into `paper/figures/`,
produces LaTeX-ready table snippets, and pushes final checkpoints + model cards to HF Hub.


In [ ]:
import sys, os; sys.path.insert(0, str(os.path.abspath(os.path.join(os.getcwd(), '..'))))
import json
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from dyna_grpo.config import PATHS, MODEL, HUB, repo_id
from dyna_grpo.utils import save_metrics

FIG = Path('../paper/figures'); FIG.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({'figure.dpi': 110, 'font.size': 10})

In [ ]:
# Figure 1: training reward curves
fig, ax = plt.subplots(figsize=(5, 3))
for run, label, color in [('baseline_grpo', 'GRPO', 'C0'),
                            ('dyna_no_cf', 'Dyna-GRPO (no CF)', 'C1'),
                            ('dyna_grpo_full', 'Dyna-GRPO (full)', 'C2')]:
    p = Path(PATHS['logs']) / f'{run}.jsonl'
    if not p.exists(): continue
    h = json.loads(p.read_text())['history']
    steps = [r['step'] for r in h]
    rew = [r['reward_mean'] for r in h]
    ax.plot(steps, rew, label=label, color=color)
ax.set_xlabel('step'); ax.set_ylabel('mean reward'); ax.legend(frameon=False)
fig.tight_layout(); fig.savefig(FIG / 'fig_reward_curves.pdf'); plt.show()

In [ ]:
# Figure 2: wall-clock to matched reward
fig, ax = plt.subplots(figsize=(5, 3))
for run, label, color in [('baseline_grpo', 'GRPO', 'C0'),
                            ('dyna_grpo_full', 'Dyna-GRPO', 'C2')]:
    p = Path(PATHS['logs']) / f'{run}.jsonl'
    if not p.exists(): continue
    h = json.loads(p.read_text())['history']
    wt = [r.get('wallclock_s', i*60) / 3600 for i, r in enumerate(h)]
    rew = [r['reward_mean'] for r in h]
    ax.plot(wt, rew, label=label, color=color)
ax.set_xlabel('wall-clock (hours)'); ax.set_ylabel('mean reward'); ax.legend(frameon=False)
fig.tight_layout(); fig.savefig(FIG / 'fig_wallclock.pdf'); plt.show()

In [ ]:
# Table 1 (LaTeX): pass@1 and TS-KL across methods
eval_path = Path(PATHS['logs']) / 'eval_results.json'
ts_path = Path(PATHS['logs']) / 'ts_kl.json'
evalr = json.loads(eval_path.read_text()) if eval_path.exists() else {}
tskl = json.loads(ts_path.read_text()) if ts_path.exists() else {}
rows = []
for name in ('zero_shot','baseline_grpo','dyna_no_cf','dyna_grpo_full'):
    e = evalr.get(name, {})
    rows.append({
        'method': name,
        'aime24': f"{e.get('aime_2024',0)*100:.1f}",
        'aime25': f"{e.get('aime_2025',0)*100:.1f}",
        'gpqa': f"{e.get('gpqa',0)*100:.1f}",
        'lcb': f"{e.get('lcb',0)*100:.1f}",
        'ts_kl': f"{tskl.get(name,0):.3f}",
    })
tex = '\\begin{tabular}{lccccc}\\toprule\nMethod & AIME-24 & AIME-25 & GPQA & LCB-v6 & TS-KL$\\downarrow$\\\\\\midrule\n'
for r in rows:
    tex += f"{r['method'].replace('_',' ')} & {r['aime24']} & {r['aime25']} & {r['gpqa']} & {r['lcb']} & {r['ts_kl']}\\\\\n"
tex += '\\bottomrule\\end{tabular}\n'
(Path('../paper') / 'table_main.tex').write_text(tex)
print(tex)

In [ ]:
# Populate paper placeholders
import re
tex_path = Path('../paper/main.tex')
src = tex_path.read_text()
subs = {
    'PASS_AIME24_GRPO': rows[1]['aime24'] if len(rows) > 1 else 'TBD',
    'PASS_AIME24_DYNA': rows[3]['aime24'] if len(rows) > 3 else 'TBD',
    'PASS_AIME25_DYNA': rows[3]['aime25'] if len(rows) > 3 else 'TBD',
    'PASS_LCB_DYNA':    rows[3]['lcb']    if len(rows) > 3 else 'TBD',
    'PASS_GPQA_DYNA':   rows[3]['gpqa']   if len(rows) > 3 else 'TBD',
    'TSKL_GRPO': f"{tskl.get('baseline_grpo',0):.3f}",
    'TSKL_DYNA': f"{tskl.get('dyna_grpo_full',0):.3f}",
}
for k, v in subs.items():
    src = src.replace(f'\\PLACEHOLDER{{{k}}}', str(v))
tex_path.write_text(src)
print('Paper placeholders filled:', list(subs.keys()))

## HF Hub push (set `PUSH_TO_HUB=1` and `HF_TOKEN=...` in env first)

In [ ]:
from dyna_grpo.hub import push
if HUB.push_to_hub:
    push(Path(PATHS['ckpts']) / 'baseline_grpo' / 'final',
         'grpo-baseline-qwen3-4b', MODEL.actor_name,
         'Vanilla GRPO baseline LoRA adapter for Dyna-GRPO comparison.',
         metrics={'AIME-24': rows[1]['aime24'], 'AIME-25': rows[1]['aime25']})
    push(Path(PATHS['ckpts']) / 'dyna_grpo_full' / 'final',
         'dyna-grpo-qwen3-4b-final', MODEL.actor_name,
         'Dyna-GRPO full method (mixed rollouts + counterfactual credit).',
         metrics={'AIME-24': rows[3]['aime24'], 'AIME-25': rows[3]['aime25'],
                  'TS-KL': f"{tskl.get('dyna_grpo_full',0):.3f}"})
    for tool in ('calc','code','search'):
        push(Path(PATHS['ckpts']) / f'predictor_{tool}',
             f'dyna-grpo-tool-predictor-{tool}', MODEL.predictor_base,
             f'{tool} world-model predictor for Dyna-GRPO.', metrics={})
else:
    print('PUSH_TO_HUB=0; skipping. Set env vars and re-run this cell to publish.')